In [1]:
# ===============================
# 1. Importar librerías
# ===============================

from pathlib import Path
from time import sleep
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
ruta = r"C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\silver\df_tilos_limpio.parquet"
df_tilos = pd.read_parquet(ruta)

In [3]:
print("Filas:", df_tilos.shape[0])
print("Columnas:", df_tilos.shape[1])
print(df_tilos.columns.tolist())

Filas: 12589
Columnas: 28
['licitacion_id', 'titulo', 'detail_url', 'updated', 'expediente', 'tipo_contrato_codigo', 'lugar_ejecucion_codigo', 'cpv_codes', 'organo_contratacion', 'estado_codigo', 'fecha_publicacion', 'procedimiento_codigo', 'importe_sin_impuestos', 'fuente_publicacion', 'presentacion_hasta', 'presentacion_hora', 'notice_types', 'organo_dir3', 'contrato_duracion', 'contrato_duracion_unidad', 'ofertas_recibidas', 'adjudicatario', 'adjudicatario_nif', 'estado', 'tipo_contrato', 'url', 'cpv_descripcion', 'cpv_nivel']


In [4]:
df_tilos.columns

Index(['licitacion_id', 'titulo', 'detail_url', 'updated', 'expediente',
       'tipo_contrato_codigo', 'lugar_ejecucion_codigo', 'cpv_codes',
       'organo_contratacion', 'estado_codigo', 'fecha_publicacion',
       'procedimiento_codigo', 'importe_sin_impuestos', 'fuente_publicacion',
       'presentacion_hasta', 'presentacion_hora', 'notice_types',
       'organo_dir3', 'contrato_duracion', 'contrato_duracion_unidad',
       'ofertas_recibidas', 'adjudicatario', 'adjudicatario_nif', 'estado',
       'tipo_contrato', 'url', 'cpv_descripcion', 'cpv_nivel'],
      dtype='object')

In [5]:
# ===============================
# 2. Frecuencia de estados
# ===============================

df_tilos["estado"].value_counts(dropna=False)

estado
Adjudicada        11806
Publicada           351
Resuelta            346
Evaluacion           80
Anuncio previo        4
Anulada               2
Name: count, dtype: int64

In [6]:
# ===============================
# Filtrar licitaciones no cerradas por estado
# ===============================

estados_no_cerrados = [
    "Publicada",
    "Anuncio previo"
]

df_tilos_no_cerradas_estado = df_tilos[
    df_tilos["estado"].isin(estados_no_cerrados)
].copy()

print("Total base original:", df_tilos.shape[0])
print("No cerradas por estado:", df_tilos_no_cerradas_estado.shape[0])

print("\nDistribución de estados filtrados:")
print(df_tilos_no_cerradas_estado["estado"].value_counts(dropna=False))

Total base original: 12589
No cerradas por estado: 355

Distribución de estados filtrados:
estado
Publicada         351
Anuncio previo      4
Name: count, dtype: int64


In [7]:
# ===============================
# Revisar fechas de presentación en no cerradas por estado
# ===============================

df_tilos_no_cerradas_estado["presentacion_hasta_dt"] = pd.to_datetime(
    df_tilos_no_cerradas_estado["presentacion_hasta"],
    errors="coerce",
    dayfirst=True
)

hoy = pd.Timestamp.today().normalize()

df_tilos_no_cerradas_estado["fecha_vigente"] = (
    df_tilos_no_cerradas_estado["presentacion_hasta_dt"] >= hoy
)

print("Fecha de corte:", hoy.date())

print("\nDistribución fecha vigente:")
print(df_tilos_no_cerradas_estado["fecha_vigente"].value_counts(dropna=False))

Fecha de corte: 2026-06-13

Distribución fecha vigente:
fecha_vigente
False    354
True       1
Name: count, dtype: int64


In [8]:
# ===============================
# 1. Crear portal solo en licitaciones no cerradas por estado
# ===============================

def detectar_portal(detail_url):
    """
    Identifica el portal de contratación a partir de la URL.
    """
    if pd.isna(detail_url):
        return "sin_url"

    detail_url = str(detail_url).lower()

    if "contratosdegalicia.gal" in detail_url:
        return "galicia"

    if "contratos-publicos.comunidad.madrid" in detail_url:
        return "madrid"

    if "contrataciondelestado.es" in detail_url:
        return "contratacion_estado"

    if "jcyl.es" in detail_url or "contratacion.jcyl" in detail_url:
        return "castilla_leon"

    return "otro"


df_tilos_no_cerradas_estado = df_tilos_no_cerradas_estado.copy()

df_tilos_no_cerradas_estado["portal"] = (
    df_tilos_no_cerradas_estado["detail_url"]
    .apply(detectar_portal)
)

print("Filas filtradas por estado:", df_tilos_no_cerradas_estado.shape[0])

df_tilos_no_cerradas_estado["portal"].value_counts(dropna=False)

Filas filtradas por estado: 355


portal
otro             249
madrid            66
galicia           39
castilla_leon      1
Name: count, dtype: int64

In [9]:
# ===============================
# 1. Filtrar registros clasificados como "otro"
# ===============================

df_otro_estado = df_tilos_no_cerradas_estado[
    df_tilos_no_cerradas_estado["portal"] == "otro"
].copy()

print("Registros clasificados como otro:", df_otro_estado.shape[0])

Registros clasificados como otro: 249


In [10]:
# ===============================
# 2. Extraer dominio de detail_url
# ===============================

from urllib.parse import urlparse

df_otro_estado["dominio_url"] = df_otro_estado["detail_url"].apply(
    lambda x: urlparse(str(x)).netloc.lower() if pd.notna(x) else "sin_url"
)

tabla_dominios_otro = (
    df_otro_estado["dominio_url"]
    .value_counts(dropna=False)
    .reset_index()
)

tabla_dominios_otro.columns = ["dominio_url", "n"]

tabla_dominios_otro["porcentaje"] = (
    tabla_dominios_otro["n"] / tabla_dominios_otro["n"].sum() * 100
).round(2)

display(tabla_dominios_otro)

,dominio_url,n,porcentaje
0,www.juntadeandalucia.es,89,35.74
1,www.contratacion.euskadi.eus,85,34.14
2,www.madrid.org,35,14.06
3,apps.euskadi.eus,14,5.62
4,www.larioja.org,8,3.21
5,hacienda.navarra.es,7,2.81
6,contractaciopublica.gencat.cat,4,1.61
7,www.carm.es,3,1.20
8,contractaciopublica.cat,3,1.20
9,www.gobiernodecanarias.org,1,0.40


In [16]:
# ===============================
# Crear dominio_url si no existe
# ===============================

from urllib.parse import urlparse
import pandas as pd

df_tilos_no_cerradas_estado = df_tilos_no_cerradas_estado.copy()

df_tilos_no_cerradas_estado["dominio_url"] = (
    df_tilos_no_cerradas_estado["detail_url"]
    .apply(lambda x: urlparse(str(x)).netloc.lower() if pd.notna(x) else "sin_url")
)

print(df_tilos_no_cerradas_estado.columns.tolist())

df_tilos_no_cerradas_estado["dominio_url"].value_counts(dropna=False)

['licitacion_id', 'titulo', 'detail_url', 'updated', 'expediente', 'tipo_contrato_codigo', 'lugar_ejecucion_codigo', 'cpv_codes', 'organo_contratacion', 'estado_codigo', 'fecha_publicacion', 'procedimiento_codigo', 'importe_sin_impuestos', 'fuente_publicacion', 'presentacion_hasta', 'presentacion_hora', 'notice_types', 'organo_dir3', 'contrato_duracion', 'contrato_duracion_unidad', 'ofertas_recibidas', 'adjudicatario', 'adjudicatario_nif', 'estado', 'tipo_contrato', 'url', 'cpv_descripcion', 'cpv_nivel', 'presentacion_hasta_dt', 'fecha_vigente', 'portal', 'dominio_url']


dominio_url
www.juntadeandalucia.es                89
www.contratacion.euskadi.eus           85
contratos-publicos.comunidad.madrid    66
www.contratosdegalicia.gal             39
www.madrid.org                         35
apps.euskadi.eus                       14
www.larioja.org                         8
hacienda.navarra.es                     7
contractaciopublica.gencat.cat          4
www.carm.es                             3
contractaciopublica.cat                 3
contratacion.jcyl.es                    1
www.gobiernodecanarias.org              1
Name: count, dtype: int64

In [17]:
# ===============================
# Filtrar licitaciones del dominio Junta de Andalucía
# ===============================

df_andalucia_estado = df_tilos_no_cerradas_estado[
    df_tilos_no_cerradas_estado["dominio_url"] == "www.juntadeandalucia.es"
].copy()

print("Licitaciones de Junta de Andalucía:", df_andalucia_estado.shape[0])

columnas_revision = [
    "licitacion_id",
    "titulo",
    "estado",
    "fecha_publicacion",
    "presentacion_hasta",
    "lugar_ejecucion_codigo",
    "organo_contratacion",
    "cpv_descripcion",
    "detail_url"
]

columnas_existentes = [
    col for col in columnas_revision
    if col in df_andalucia_estado.columns
]

display(
    df_andalucia_estado[columnas_existentes]
    .reset_index(drop=True)
)

Licitaciones de Junta de Andalucía: 89


,licitacion_id,titulo,estado,fecha_publicacion,presentacion_hasta,lugar_ejecucion_codigo,organo_contratacion,cpv_descripcion,detail_url
0,2303b2fc43d597b7,Servicio de podología para las personas socias...,Publicada,2018-02-26,2018-03-12,ES618,"Delegación Territorial de Igualdad, Salud y Po...",Servicios de salud,http://www.juntadeandalucia.es/temas/contratac...
1,fe0900f972a13d9b,2018/012745. Servicio de terapias oncológicas ...,Publicada,2018-02-27,2018-03-06,ES614,Servicio Andaluz de Salud,Servicios de salud,http://www.juntadeandalucia.es/temas/contratac...
2,5ecbd8fc224956aa,"2018/013170 Servicio para la realización, en l...",Publicada,2018-03-06,2018-04-10,ES613,Servicio Andaluz de Salud,Servicios prestados por laboratorios médicos,http://www.juntadeandalucia.es/temas/contratac...
3,1a69f3b8bef4f05b,servicio de prevención ajeno de riesgos labora...,Publicada,2018-03-07,2018-03-21,ES618,Agencia Andaluza de la Energía,Servicios de salud,http://www.juntadeandalucia.es/temas/contratac...
4,233d71c00c702d76,2018/014757 Servicios de asistencia sanitaria ...,Publicada,2018-03-08,2018-04-16,ES618,Servicio Andaluz de Salud,Servicios de salud,http://www.juntadeandalucia.es/temas/contratac...
...,...,...,...,...,...,...,...,...,...
84,544f9ef4d8816581,servicio sanitario para el personal del plan i...,Publicada,2026-03-23,2026-04-13,ES61,Agencia de Seguridad y Gestión Integral de Eme...,Servicios de salud,https://www.juntadeandalucia.es/haciendayadmin...
85,d6b2ef2a4bd53f8b,0000084/2025 servicio de asistencia sanitaria ...,Publicada,2026-03-30,2026-04-15,ES614,Servicio Andaluz de Salud,Servicios de salud,https://www.juntadeandalucia.es/haciendayadmin...
86,6c9683d02219cedf,paam 53/2026 (sevilla) acuerdo marco con varia...,Publicada,2026-04-29,2026-06-01,ES618,Servicio Andaluz de Salud,Servicios de salud,https://www.juntadeandalucia.es/haciendayadmin...
87,41a641fa8ad1a1bc,servicios médicos sanitarios de la copa del mu...,Publicada,2026-04-30,2026-05-15,ES618,Empresa Pública para la Gestión del Turismo y ...,Servicios de ejercicio de la medicina y servic...,https://www.juntadeandalucia.es/haciendayadmin...


In [18]:
# ===============================
# 1. Lugar de ejecución en licitaciones filtradas por estado
# ===============================

tabla_lugar_estado = (
    df_tilos_no_cerradas_estado["lugar_ejecucion_codigo"]
    .value_counts(dropna=False)
    .reset_index()
)

tabla_lugar_estado.columns = ["lugar_ejecucion_codigo", "n"]

tabla_lugar_estado["porcentaje"] = (
    tabla_lugar_estado["n"] / tabla_lugar_estado["n"].sum() * 100
).round(2)

display(tabla_lugar_estado)

,lugar_ejecucion_codigo,n,porcentaje
0,ES300,99,27.89
1,ES213,32,9.01
2,ES211,27,7.61
3,ES61,26,7.32
4,ES618,20,5.63
5,ES21,20,5.63
6,ES212,20,5.63
7,ES11,14,3.94
8,ES111,12,3.38
9,ES614,11,3.10


In [20]:
# ===============================
# 2. Dominio URL vs lugar de ejecución
# ===============================
tabla_dominio_lugar = (
    df_tilos_no_cerradas_estado
    .groupby(["dominio_url", "lugar_ejecucion_codigo"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(by="n", ascending=False)
    .reset_index(drop=True)
)

display(tabla_dominio_lugar)

,dominio_url,lugar_ejecucion_codigo,n
0,contratos-publicos.comunidad.madrid,ES300,64
1,www.madrid.org,ES300,35
2,www.contratacion.euskadi.eus,ES213,30
3,www.juntadeandalucia.es,ES61,26
4,www.contratacion.euskadi.eus,ES211,22
5,www.juntadeandalucia.es,ES618,20
6,www.contratacion.euskadi.eus,ES212,19
7,www.contratosdegalicia.gal,ES11,14
8,www.contratacion.euskadi.eus,ES21,14
9,www.contratosdegalicia.gal,ES111,12


In [21]:
# ===============================
# Seleccionar máximo 20 licitaciones
# Nacionales o cercanas a Segovia
# ===============================

def clasificar_cercania_segovia(codigo):
    """
    Clasifica cercanía geográfica a Segovia usando lugar_ejecucion_codigo.
    """
    if pd.isna(codigo):
        return "revisar_sin_codigo"

    codigo = str(codigo).upper().strip()

    # Nacional / España agregado
    if codigo in ["ES", "ES0", "ES00"]:
        return "nacional"

    # Segovia
    if codigo == "ES416":
        return "segovia"

    # Castilla y León
    if codigo.startswith("ES41"):
        return "castilla_leon"

    # Madrid
    if codigo == "ES300" or codigo.startswith("ES30"):
        return "madrid"

    return "fuera_zona"


df_tilos_no_cerradas_estado["cercania_segovia"] = (
    df_tilos_no_cerradas_estado["lugar_ejecucion_codigo"]
    .apply(clasificar_cercania_segovia)
)

prioridades_validas = [
    "nacional",
    "segovia",
    "castilla_leon",
    "madrid",
    "revisar_sin_codigo"
]

df_candidatas_segovia = df_tilos_no_cerradas_estado[
    df_tilos_no_cerradas_estado["cercania_segovia"].isin(prioridades_validas)
].copy()

print("Candidatas nacionales o cercanas a Segovia:", df_candidatas_segovia.shape[0])

df_candidatas_segovia_20 = (
    df_candidatas_segovia
    .sort_values(
        by=["cercania_segovia", "presentacion_hasta"],
        ascending=[True, True]
    )
    .head(20)
    .reset_index(drop=True)
)

columnas_revision = [
    "licitacion_id",
    "titulo",
    "estado",
    "fecha_publicacion",
    "presentacion_hasta",
    "dominio_url",
    "lugar_ejecucion_codigo",
    "cercania_segovia",
    "organo_contratacion",
    "cpv_descripcion",
    "detail_url"
]

columnas_existentes = [
    col for col in columnas_revision
    if col in df_candidatas_segovia_20.columns
]

display(df_candidatas_segovia_20[columnas_existentes])

Candidatas nacionales o cercanas a Segovia: 103


,licitacion_id,titulo,estado,fecha_publicacion,presentacion_hasta,dominio_url,lugar_ejecucion_codigo,cercania_segovia,organo_contratacion,cpv_descripcion,detail_url
0,69f1c717a376311d,asistencia sanitaria diagnóstico por resonanci...,Publicada,2018-03-05,2018-03-28,contratacion.jcyl.es,ES414,castilla_leon,Gerencia de Atención Especializada de Palencia,Servicios de salud,https://contratacion.jcyl.es/web/jcyl/Contrata...
1,4cbe6c463d894a88,Servicios sanitarios para la temporada de vera...,Publicada,2018-02-23,2018-03-12,www.madrid.org,ES300,madrid,"Consejería de Cultura, Turismo y Deportes",Servicios prestados por enfermeros,http://www.madrid.org/cs/Satellite?op2=PCON&id...
2,075dac71258ec192,Servicio de análisis e informe de resultados d...,Publicada,2018-03-01,2018-03-14,www.madrid.org,ES300,madrid,Hospital Universitario del Sureste,Servicios prestados por laboratorios médicos,http://www.madrid.org/cs/Satellite?op2=PCON&id...
3,a765d556e7c58ca4,"Servicio de: extracción, traslado, destrucción...",Publicada,2018-03-13,2018-03-26,www.madrid.org,ES300,madrid,Servicio Madrileño de Salud,Servicios varios de salud,http://www.madrid.org/cs/Satellite?op2=PCON&id...
4,ef5caa38bddf3488,Servicio de laboratorio de análisis clínicos p...,Publicada,2018-07-23,2018-08-14,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, Sociedad A...",Servicios prestados por laboratorios médicos,http://www.madrid.org/cs/Satellite?op2=PCON&id...
5,72cb60c324ca8f34,Servicio de laboratorio para la realización de...,Publicada,2018-09-19,2018-10-08,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios prestados por laboratorios médicos,http://www.madrid.org/cs/Satellite?op2=PCON&id...
6,da1fec228406a862,Contratación de un servicio médico de neumolog...,Publicada,2018-09-20,2018-10-10,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios de médicos especialistas,http://www.madrid.org/cs/Satellite?op2=PCON&id...
7,2c2b7a36be594b70,Contratación de un servicio médico para la rea...,Publicada,2018-09-18,2018-10-11,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios de médicos especialistas,http://www.madrid.org/cs/Satellite?op2=PCON&id...
8,20851c82c510177f,Servicio médico en la especialidad de Cirugía ...,Publicada,2018-09-18,2018-10-18,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios de médicos especialistas,http://www.madrid.org/cs/Satellite?op2=PCON&id...
9,53bcbf153dc11a56,Servicio médico para la realización de consult...,Publicada,2018-10-11,2018-11-05,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios de médicos especialistas,http://www.madrid.org/cs/Satellite?op2=PCON&id...


In [22]:
# ===============================
# Ordenar por prioridad geográfica real
# ===============================

orden_prioridad = {
    "segovia": 1,
    "castilla_leon": 2,
    "madrid": 3,
    "nacional": 4,
    "revisar_sin_codigo": 5
}

df_candidatas_segovia["orden_prioridad_geo"] = (
    df_candidatas_segovia["cercania_segovia"]
    .map(orden_prioridad)
)

df_candidatas_segovia_20 = (
    df_candidatas_segovia
    .sort_values(
        by=["orden_prioridad_geo", "presentacion_hasta"],
        ascending=[True, True]
    )
    .head(20)
    .reset_index(drop=True)
)

display(df_candidatas_segovia_20[columnas_existentes])

,licitacion_id,titulo,estado,fecha_publicacion,presentacion_hasta,dominio_url,lugar_ejecucion_codigo,cercania_segovia,organo_contratacion,cpv_descripcion,detail_url
0,69f1c717a376311d,asistencia sanitaria diagnóstico por resonanci...,Publicada,2018-03-05,2018-03-28,contratacion.jcyl.es,ES414,castilla_leon,Gerencia de Atención Especializada de Palencia,Servicios de salud,https://contratacion.jcyl.es/web/jcyl/Contrata...
1,4cbe6c463d894a88,Servicios sanitarios para la temporada de vera...,Publicada,2018-02-23,2018-03-12,www.madrid.org,ES300,madrid,"Consejería de Cultura, Turismo y Deportes",Servicios prestados por enfermeros,http://www.madrid.org/cs/Satellite?op2=PCON&id...
2,075dac71258ec192,Servicio de análisis e informe de resultados d...,Publicada,2018-03-01,2018-03-14,www.madrid.org,ES300,madrid,Hospital Universitario del Sureste,Servicios prestados por laboratorios médicos,http://www.madrid.org/cs/Satellite?op2=PCON&id...
3,a765d556e7c58ca4,"Servicio de: extracción, traslado, destrucción...",Publicada,2018-03-13,2018-03-26,www.madrid.org,ES300,madrid,Servicio Madrileño de Salud,Servicios varios de salud,http://www.madrid.org/cs/Satellite?op2=PCON&id...
4,ef5caa38bddf3488,Servicio de laboratorio de análisis clínicos p...,Publicada,2018-07-23,2018-08-14,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, Sociedad A...",Servicios prestados por laboratorios médicos,http://www.madrid.org/cs/Satellite?op2=PCON&id...
5,72cb60c324ca8f34,Servicio de laboratorio para la realización de...,Publicada,2018-09-19,2018-10-08,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios prestados por laboratorios médicos,http://www.madrid.org/cs/Satellite?op2=PCON&id...
6,da1fec228406a862,Contratación de un servicio médico de neumolog...,Publicada,2018-09-20,2018-10-10,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios de médicos especialistas,http://www.madrid.org/cs/Satellite?op2=PCON&id...
7,2c2b7a36be594b70,Contratación de un servicio médico para la rea...,Publicada,2018-09-18,2018-10-11,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios de médicos especialistas,http://www.madrid.org/cs/Satellite?op2=PCON&id...
8,20851c82c510177f,Servicio médico en la especialidad de Cirugía ...,Publicada,2018-09-18,2018-10-18,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios de médicos especialistas,http://www.madrid.org/cs/Satellite?op2=PCON&id...
9,53bcbf153dc11a56,Servicio médico para la realización de consult...,Publicada,2018-10-11,2018-11-05,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios de médicos especialistas,http://www.madrid.org/cs/Satellite?op2=PCON&id...


In [23]:
df_candidatas_segovia["cercania_segovia"].value_counts(dropna=False)

cercania_segovia
madrid           101
castilla_leon      1
nacional           1
Name: count, dtype: int64

In [24]:
# ===============================
# Filtrar 20 candidatas:
# 18 Madrid + 2 restantes no Madrid
# ===============================

# Base: candidatas nacionales o cercanas a Segovia
df_base_candidatas = df_candidatas_segovia.copy()

# Asegurar orden por fecha de presentación
df_base_candidatas["presentacion_hasta_dt"] = pd.to_datetime(
    df_base_candidatas["presentacion_hasta"],
    errors="coerce",
    dayfirst=True
)

# -------------------------------
# 1. Seleccionar 18 de Madrid
# -------------------------------

df_madrid_18 = (
    df_base_candidatas[
        df_base_candidatas["cercania_segovia"] == "madrid"
    ]
    .sort_values("presentacion_hasta_dt", ascending=True)
    .head(18)
    .copy()
)

# -------------------------------
# 2. Seleccionar 2 no Madrid
#    Prioridad: Castilla y León, luego Nacional
# -------------------------------

orden_no_madrid = {
    "castilla_leon": 1,
    "nacional": 2
}

df_no_madrid = df_base_candidatas[
    df_base_candidatas["cercania_segovia"].isin(["castilla_leon", "nacional"])
].copy()

df_no_madrid["orden_no_madrid"] = (
    df_no_madrid["cercania_segovia"]
    .map(orden_no_madrid)
)

df_no_madrid_2 = (
    df_no_madrid
    .sort_values(
        by=["orden_no_madrid", "presentacion_hasta_dt"],
        ascending=[True, True]
    )
    .head(2)
    .copy()
)

# -------------------------------
# 3. Unir selección final
# -------------------------------

df_muestra_20_segovia = (
    pd.concat(
        [df_madrid_18, df_no_madrid_2],
        ignore_index=True
    )
    .reset_index(drop=True)
)

print("Total seleccionadas:", df_muestra_20_segovia.shape[0])

print("\nDistribución por cercanía:")
print(df_muestra_20_segovia["cercania_segovia"].value_counts(dropna=False))

# -------------------------------
# 4. Visualizar resultado
# -------------------------------

columnas_revision = [
    "licitacion_id",
    "titulo",
    "estado",
    "fecha_publicacion",
    "presentacion_hasta",
    "presentacion_hasta_dt",
    "dominio_url",
    "lugar_ejecucion_codigo",
    "cercania_segovia",
    "organo_contratacion",
    "cpv_descripcion",
    "detail_url"
]

columnas_existentes = [
    col for col in columnas_revision
    if col in df_muestra_20_segovia.columns
]

display(df_muestra_20_segovia[columnas_existentes])

Total seleccionadas: 20

Distribución por cercanía:
cercania_segovia
madrid           18
castilla_leon     1
nacional          1
Name: count, dtype: int64


,licitacion_id,titulo,estado,fecha_publicacion,presentacion_hasta,presentacion_hasta_dt,dominio_url,lugar_ejecucion_codigo,cercania_segovia,organo_contratacion,cpv_descripcion,detail_url
0,4cbe6c463d894a88,Servicios sanitarios para la temporada de vera...,Publicada,2018-02-23,2018-03-12,2018-03-12,www.madrid.org,ES300,madrid,"Consejería de Cultura, Turismo y Deportes",Servicios prestados por enfermeros,http://www.madrid.org/cs/Satellite?op2=PCON&id...
1,075dac71258ec192,Servicio de análisis e informe de resultados d...,Publicada,2018-03-01,2018-03-14,2018-03-14,www.madrid.org,ES300,madrid,Hospital Universitario del Sureste,Servicios prestados por laboratorios médicos,http://www.madrid.org/cs/Satellite?op2=PCON&id...
2,a765d556e7c58ca4,"Servicio de: extracción, traslado, destrucción...",Publicada,2018-03-13,2018-03-26,2018-03-26,www.madrid.org,ES300,madrid,Servicio Madrileño de Salud,Servicios varios de salud,http://www.madrid.org/cs/Satellite?op2=PCON&id...
3,ef5caa38bddf3488,Servicio de laboratorio de análisis clínicos p...,Publicada,2018-07-23,2018-08-14,2018-08-14,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, Sociedad A...",Servicios prestados por laboratorios médicos,http://www.madrid.org/cs/Satellite?op2=PCON&id...
4,72cb60c324ca8f34,Servicio de laboratorio para la realización de...,Publicada,2018-09-19,2018-10-08,2018-10-08,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios prestados por laboratorios médicos,http://www.madrid.org/cs/Satellite?op2=PCON&id...
5,da1fec228406a862,Contratación de un servicio médico de neumolog...,Publicada,2018-09-20,2018-10-10,2018-10-10,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios de médicos especialistas,http://www.madrid.org/cs/Satellite?op2=PCON&id...
6,2c2b7a36be594b70,Contratación de un servicio médico para la rea...,Publicada,2018-09-18,2018-10-11,2018-10-11,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios de médicos especialistas,http://www.madrid.org/cs/Satellite?op2=PCON&id...
7,20851c82c510177f,Servicio médico en la especialidad de Cirugía ...,Publicada,2018-09-18,2018-10-18,2018-10-18,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios de médicos especialistas,http://www.madrid.org/cs/Satellite?op2=PCON&id...
8,53bcbf153dc11a56,Servicio médico para la realización de consult...,Publicada,2018-10-11,2018-11-05,2018-11-05,www.madrid.org,ES300,madrid,"Empresa Pública de Metro de Madrid, S.A.",Servicios de médicos especialistas,http://www.madrid.org/cs/Satellite?op2=PCON&id...
9,f748bce950d2b1bb,Análisis para la obtención del estado de situa...,Publicada,2018-10-24,2018-11-08,2018-11-08,www.madrid.org,ES300,madrid,Consejería de Sanidad,Servicios prestados por laboratorios médicos,http://www.madrid.org/cs/Satellite?op2=PCON&id...


In [25]:
# ===============================
# Ver solo las URLs
# ===============================

display(
    df_muestra_20_segovia[["detail_url"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

,detail_url
0,http://www.madrid.org/cs/Satellite?op2=PCON&id...
1,http://www.madrid.org/cs/Satellite?op2=PCON&id...
2,http://www.madrid.org/cs/Satellite?op2=PCON&id...
3,http://www.madrid.org/cs/Satellite?op2=PCON&id...
4,http://www.madrid.org/cs/Satellite?op2=PCON&id...
5,http://www.madrid.org/cs/Satellite?op2=PCON&id...
6,http://www.madrid.org/cs/Satellite?op2=PCON&id...
7,http://www.madrid.org/cs/Satellite?op2=PCON&id...
8,http://www.madrid.org/cs/Satellite?op2=PCON&id...
9,http://www.madrid.org/cs/Satellite?op2=PCON&id...


In [26]:
# ===============================
# Guardar muestra de 20 en capa Silver
# ===============================

from pathlib import Path
import pandas as pd

ruta_silver = Path(
    r"C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\Silver"
)

ruta_salida = ruta_silver / "muestra_20_segovia_18_madrid_2_restantes.parquet"

df_muestra_20_segovia.to_parquet(
    ruta_salida,
    index=False
)

print("Archivo parquet guardado correctamente.")
print("Ruta:", ruta_salida)
print("Filas guardadas:", df_muestra_20_segovia.shape[0])
print("Columnas guardadas:", df_muestra_20_segovia.shape[1])

Archivo parquet guardado correctamente.
Ruta: C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\Silver\muestra_20_segovia_18_madrid_2_restantes.parquet
Filas guardadas: 20
Columnas guardadas: 35
